## Spark SQL and DataFrame Overview

Purpose: To provide a basic overview of interacting with Spark, Spark SQL and Dataframes

In [1]:
import pandas as pd
import pyspark
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql import types
import warnings
warnings.filterwarnings('ignore', category=UserWarning)


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/04 15:02:30 WARN Utils: Your hostname, mach48192, resolves to a loopback address: 127.0.1.1; using 192.168.50.2 instead (on interface eno1)
26/03/04 15:02:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 15:02:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-03 16:17:11--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T22%3A13%3A00Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-03T21%3A12%3A52Z&ske=2026-03-03T22%3A13%3A00Z&sks=b&skv=2018-11-09&sig=B1CDgxOH4oL5fhECThcwYuHQW1ezBS4sXHFAOeja0Pc%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjU3NjIzMSwibmJmIjoxNzcyNTcyNjMxLCJwYXRoIjoi

In [ ]:
# Run this in terminal rather than jupyter notebook because it's too slow
!gzip -dc fhvhv_tripdata_2021-01.csv.gz > fhvhv_tripdata_2021-01.csv

In [6]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [4]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [5]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [8]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [6]:
df_pandas = pd.read_csv('head.csv')

In [7]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [8]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [9]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [10]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [11]:
# Repartition does actually happen at this point
df = df.repartition(24)

In [ ]:
# Once you do something with the dataframe, is when the repartition occurs
df.write.parquet('fhvhv/2021/01/')

26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/03 16:43:45 WARN MemoryManager: Total allocation exceeds 95.

In [12]:
df = spark.read.parquet('fhvhv/2021/01/')

In [13]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [14]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0005|              B02510|2021-01-01 00:40:19|2021-01-01 01:04:35|         142|          80|   NULL|
|           HV0003|              B02869|2021-01-01 14:39:29|2021-01-01 14:59:45|         148|          68|   NULL|
|           HV0005|              B02510|2021-01-01 05:19:33|2021-01-01 05:27:55|          17|          17|   NULL|
|           HV0003|              B02887|2021-01-01 02:28:51|2021-01-01 02:38:51|         168|         212|   NULL|
|           HV0005|              B02510|2021-01-01 10:10:56|2021-01-01 10:16:51|         116|         244|   NULL|
|           HV0003|              B02864|2021-01-01 03:59:03|2021-01-01 04:09:18|

In [15]:
# Filtering spark dataframe

df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [16]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


### User Defined Function Example

In [18]:
from pyspark.sql import functions as F

In [20]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
        .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-01 00:40:19|2021-01-01 01:04:35|         142|          80|   NULL| 2021-01-01|  2021-01-01|
|           HV0003|              B02869|2021-01-01 14:39:29|2021-01-01 14:59:45|         148|          68|   NULL| 2021-01-01|  2021-01-01|
|           HV0005|              B02510|2021-01-01 05:19:33|2021-01-01 05:27:55|          17|          17|   NULL| 2021-01-01|  2021-01-01|
|           HV0003|              B02887|2021-01-01 02:28:51|2021-01-01 02:38:51|         168|         212|   NULL| 2021-01-01|  2021-01-01|
|           HV0005| 

In [21]:
# Another udf example
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [22]:
crazy_stuff('B02884')

's/b44'

In [ ]:
# Creates custom udf function
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [24]:
# Using udf function
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
        .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
        .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|base_id|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+-------+
|           HV0005|              B02510|2021-01-01 00:40:19|2021-01-01 01:04:35|         142|          80|   NULL| 2021-01-01|  2021-01-01|  e/9ce|
|           HV0003|              B02869|2021-01-01 14:39:29|2021-01-01 14:59:45|         148|          68|   NULL| 2021-01-01|  2021-01-01|  e/b35|
|           HV0005|              B02510|2021-01-01 05:19:33|2021-01-01 05:27:55|          17|          17|   NULL| 2021-01-01|  2021-01-01|  e/9ce|
|           HV0003|              B02887|2021-01-01 02:28:51|2021-01-01 02:38:51|         168|         212|   NUL